In [59]:
print("helloworld")

helloworld


In [60]:
import sys
!{sys.executable} -m pip install -U pip setuptools wheel
!{sys.executable} -m pip uninstall -y fasttext
!{sys.executable} -m pip install fasttext-wheel
!{sys.executable} -m pip install nltk
!{sys.executable} -m pip install openai
!{sys.executable} -m pip install transformers numpy
!{sys.executable} -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu




Looking in indexes: https://download.pytorch.org/whl/cpu


In [61]:
import os, urllib.request, gzip, shutil

NB_GZ = "numberbatch-en-19.08.txt.gz"
NB_TXT = "numberbatch-en-19.08.txt"
NB_URL = "https://conceptnet.s3.amazonaws.com/downloads/2019/numberbatch/numberbatch-en-19.08.txt.gz"

if not os.path.exists(NB_GZ) and not os.path.exists(NB_TXT):
    print("Downloading Numberbatch...")
    urllib.request.urlretrieve(NB_URL, NB_GZ)
    print("Downloaded:", NB_GZ)

if os.path.exists(NB_GZ) and not os.path.exists(NB_TXT):
    print("Extracting Numberbatch...")
    with gzip.open(NB_GZ, "rb") as f_in, open(NB_TXT, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    print("Extracted:", NB_TXT)


In [62]:
import os
import re
import numpy as np
import fasttext
import nltk
from nltk.corpus import stopwords

In [63]:


MODEL_PATH = r"C:\Users\kyabr\PersonalProjects\HackaCookers\models\cc.en.300.bin"

try:
    STOPWORDS = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords")
    STOPWORDS = set(stopwords.words("english"))

# Make sure the file exists
if not os.path.isfile(MODEL_PATH):
    raise FileNotFoundError(f"Model file not found:\n{MODEL_PATH}")

In [64]:


print("Loading fastText model (this can take a while and may use lots of RAM)...")
model = fasttext.load_model(MODEL_PATH)  # <-- no mmap on your build
print("Model loaded.")

Loading fastText model (this can take a while and may use lots of RAM)...
Model loaded.


The Caption/Bottom description relationship. It works by primairly identifying key words relating to a selected word, it does not consider context between words (it assumes we have a short sentence to make it work)

In [65]:
def tokenize(text: str) -> list[str]: #convert caption to list of strings
    text = text.lower() #put everything lowercase
    text = re.sub(r"#", " ", text) #remove hashtags"
    text = re.sub(r"[^a-z\s]", " ", text) #get rid of everything except lowercase letters and spaces

    tokens = text.split() #split string into words
    cleaned = []

    for t in tokens:
        if len(t) <= 2: #remove short words
            continue
        if t.endswith("s") and not t.endswith("ss"):
            t = t[:-1]

        if t in STOPWORDS: #get rid of stopwords
            continue
        cleaned.append(t)

    return cleaned


def cosine(a: np.ndarray, b: np.ndarray) -> float: #method of measuring semantic simularity
    denom = np.linalg.norm(a) * np.linalg.norm(b) #vector magnitude for normalization
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom) #cosine similarity

def relatedness_score(word: str, #keyword
                      caption: str, #caption
                      tau: float = 0.35, #similarity threshold
                        k: float = 12.0, #slope
                        high: float = 0.45, #how many strong matches exist
                        bonus: float = 0.05) -> float: #bonus for many strong matches
    tokens = tokenize(caption) 
    if not tokens:
        return 0.0

    q = model.get_word_vector(word) #embed the keyword
    sims = np.array([cosine(q, model.get_word_vector(tok)) for tok in tokens], dtype=np.float32) #compute similarity per token

    best = float(sims.max()) #extract most similar word
    count = int(np.sum(sims >= high)) 


 
    base = 1.0 / (1.0 + np.exp(-k * (best - tau)))
# - tau sets the similarity threshold where confidence = 0.5
# - k controls how sharply the score increases past that threshold
   

    score = base + bonus * max(0, count - 1)  # small bonus for multiple strong hits
    return float(min(1.0, score)) 


# Notebook test run:
tests = [
    ("girl", "Fortnite update just dropped and it's insane"),
    ("girl", "Females are crazy"),
    ("girl", "female minds man"),
    ("pie", "let's cook up something tonight"),
    ("pie", "it was a huge disaster"),
    ("puppy", "I want a dog"),
    ("puppy", "women are cool"),
    ("oil", "environmentalism is very important for us to work with"),
    ("oil", "I'm a blue collar guy"),
    ("oil", "I wonder what it's for dinner"),
    ("oil", "I am a strong man"),
    ("oil", "tar is good for you"),
    ("oil", "I'm a chemist in my day job"),
    ("republican", "You know the 6-7 guy? Oh it's the Monarch. It's the Monarch. Yeah, yeah. The guy that made the song, he does Antarilla. Yeah, yeah. But he wears masks. Do you see the masks? No, I didn't. And all of his music video, he wears really creepy masks. Oh, there's so much footage of him. And he even said to himself, there's pictures of him holding goat heads up. Yeah, yeah. Sacrificing animals. And he's saying his religion. It uses animal sacrifice. Yeah. There's certain gods he prays to that give him what he wants out of life. Yeah. By sacrificing or giving offerings to him, this is like real witchcraft. Yeah, that's when you say six, that's a goddess. Say seven, that's also a goddess. So what he's saying, say nothing at this. But I actually don't know where 6-7 came with that. Such a random number though. So the theory is, 6-7 is supposed to be something that we don't understand, but it conjures almost a spirit. It's like chanting onto it. It's like a name or almost like a prayer to just say, you know how we say like Hallelujah. Yeah.")]
for w, c in tests:
    score = relatedness_score(w, c)
    print(f"{w!r} vs {c!r} → score={score:.3f}")

'girl' vs "Fortnite update just dropped and it's insane" → score=0.091
'girl' vs 'Females are crazy' → score=0.828
'girl' vs 'female minds man' → score=0.970
'pie' vs "let's cook up something tonight" → score=0.353
'pie' vs 'it was a huge disaster' → score=0.113
'puppy' vs 'I want a dog' → score=0.995
'puppy' vs 'women are cool' → score=0.039
'oil' vs 'environmentalism is very important for us to work with' → score=0.174
'oil' vs "I'm a blue collar guy" → score=0.074
'oil' vs "I wonder what it's for dinner" → score=0.043
'oil' vs 'I am a strong man' → score=0.062
'oil' vs 'tar is good for you' → score=0.684
'oil' vs "I'm a chemist in my day job" → score=0.143
'republican' vs "You know the 6-7 guy? Oh it's the Monarch. It's the Monarch. Yeah, yeah. The guy that made the song, he does Antarilla. Yeah, yeah. But he wears masks. Do you see the masks? No, I didn't. And all of his music video, he wears really creepy masks. Oh, there's so much footage of him. And he even said to himself, ther

This section determines how correlated an auto generated caption is to our keyword

Methods we can test: 
1. getting a keyword for the caption and then seeing if it is close to our desired word
2. Overall sentiment analysis (trying this first)


In [66]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install sentence-transformers



Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cpu
Defaulting to user installation because normal site-packages is not writeable


In [67]:
import sys

!"{sys.executable}" -m pip install --upgrade pip
!"{sys.executable}" -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!"{sys.executable}" -m pip install sentence-transformers
!"{sys.executable}" -m pip install einops


Looking in indexes: https://download.pytorch.org/whl/cpu


In [68]:
import torch
print("Torch version:", torch.__version__)


Torch version: 2.9.1+cpu


In [69]:
import sys

!"{sys.executable}" -m pip install -U sentence-transformers transformers einops


In [70]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "nomic-ai/nomic-embed-text-v2-moe",
    trust_remote_code=True
)


In [71]:
import re
import numpy as np

def explain_by_chunks(keyword, caption, model):
    query = f"This caption is related to {keyword} politics in the United States."
    q = model.encode(query, normalize_embeddings=True)

    chunks = re.split(r'(?<=[.!?])\s+', caption)
    chunks = [c for c in chunks if c.strip()]

    chunk_embs = model.encode(chunks, normalize_embeddings=True)
    scores = chunk_embs @ q

    results = sorted(
        zip(chunks, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return results


In [72]:
explanations = explain_by_chunks(
    "republican",
    "Charlie Kirk speaks at a Turning Point USA event. Great crowd energy.",
    model
)
for text, score in explanations:
    print(f"{score:.3f} → {text}")

0.256 → Charlie Kirk speaks at a Turning Point USA event.
0.146 → Great crowd energy.


In [ ]:
import re
import numpy as np

def embed_text(texts): #encode the texts
    return model.encode(texts, normalize_embeddings=True, convert_to_numpy=True)

def split_sentences(text): #split into sentences for the algorithm
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [p for p in parts if p.strip()]

def score_and_explain(keyword, #topic
                      captions, #list of captions as a string
                      top_k=3 #how many top chunks to show as output
                      ): 
    query = f"This caption is related to {keyword}." 
    q = embed_text([query])[0]

    results = []

    for i, caption in enumerate(captions):
        # Full caption score
        cap_emb = embed_text([caption])[0]
        full_score = float(cap_emb @ q)

        # Chunk-level explanation
        chunks = split_sentences(caption)
        chunk_embs = embed_text(chunks)
        chunk_scores = chunk_embs @ q

        ranked_chunks = sorted(
            zip(chunks, chunk_scores),
            key=lambda x: x[1],
            reverse=True
        )

        results.append((full_score, ranked_chunks[:top_k]))

        # ---- PRINT OUTPUT ----
        print(f"\nCaption {i+1}")
        print(f"FINAL SCORE: {full_score:.3f}")
        print("Top contributing chunks:")
        for text, score in ranked_chunks[:top_k]:
            print(f"  {score:.3f} → {text}")

    return results

In [77]:
keyword = "republican"

captions = [
    "Charlie Kirk speaks at a Turning Point USA event. Things do not go smooth as the flag ended up falling over onto a man. I wonder if my TV will ever actually speak about this or if I will have to.",
    "New vegan recipe ideas for meal prep. Low calorie and Fat Free.",
    "6-7 your mom holy mama molly i wonder if you've ever seen anything like this",
    "The weather was absoltely insane today. It was like a hurricaine.",
    "doesn't have a plan. She copied Biden's plan and it's like four sentences like RunSpot Run. Crime in this country is through the roof...",
    "take a shower with my 14 year old son and I didn't expect him to do this to me. I'm a 37 year old woman and I know what you're thinking when you hear that. My 14 year old son takes a shower with me, right? Well, I have a good reason for that. We share the same bathroom and bedroom. At first, I hesitated to admit this online because of the judge's looks and comments that I would receive from strangers. But here it goes. It all started three years ago when my husband suddenly left us without any explanation. It was a total shock. We had been married for more than 15 years and he seemed happy until one day he packed his bags and disappeared. All he said to us was, I'm going to buy milk at the gas station. I'll be right back. He never came back. The kids were devastated, especially my youngest son Max. He clung to me more than ever, looking for comfort wherever he could find it. At night, Max would come into my room trembling with cold, asking for a warm blanket. As much as I wanted to send him back to his own room, I couldn't stand seeing him so vulnerable. So, gradually, Max got closer to my side and finally ended up sleeping next to me. Sometimes even snuggling up to me. Over time, I began to notice that Max did not move away from my side during the day either. Whether it was showering or brushing his teeth, Max was always there, watching closely. I tried to ignore him hoping he would get over that phase, like most teenage behaviors. However, as the days became weeks and then months, I realized that this was not going to disappear soon. One night, while I was showering, Max asked if he could join me. I asked him why, and he said that all the boys at school say that I have a level 9 gas. I don't know what that means, but at first I was surprised. I doubted, without being sure if it was appropriate or healthy. But seeing him so small and fragile, I knew deep down that I couldn't say no. And so our new ritual began. Everything was going well."
]

score_and_explain(keyword, captions)


NameError: name 'q' is not defined